In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import json
import os

In [2]:
compression_locations = {
    "No Compression": "uncompressed",
    "Nexus Compression": "nexus"
}

# Define the producer rates
producer_rates = [100, 1000]

# Create a dictionary to store the data
data = {
    "Compression Location": [],
    "Producer Rate": [],
    "Compression Ratio": [],
    "aggregatedPublishLatency95pct": []
}

# Iterate over the compression locations
for location, directory in compression_locations.items():
    # Construct the full directory path
    full_directory = os.path.join(directory)
    
    # Read the recorded_ratios.json file
    with open(os.path.join(full_directory, "recorded_ratios.json"), "r") as file:
        ratios = json.load(file)
        
        # Initialize lists to store the compression ratios and write latencies
        compression_ratios_100 = []
        compression_ratios_1000 = []
        write_latencies_100 = []
        write_latencies_1000 = []
        
        # Iterate over the workloads in the JSON file
        for workload in ratios.values():
            # Check the producer rate
            if workload["producerRate"] == 100:
                compression_ratios_100.append(workload["compressionRatio"])
                write_latencies_100.append(workload["aggregatedPublishLatency95pct"])
            elif workload["producerRate"] == 1000:
                compression_ratios_1000.append(workload["compressionRatio"])
                write_latencies_1000.append(workload["aggregatedPublishLatency95pct"])
        
        # Calculate the average compression ratio and write latency for each producer rate
        avg_compression_ratio_100 = sum(compression_ratios_100) / len(compression_ratios_100)
        avg_write_latency_100 = sum(write_latencies_100) / len(write_latencies_100)
        avg_compression_ratio_1000 = sum(compression_ratios_1000) / len(compression_ratios_1000)
        avg_write_latency_1000 = sum(write_latencies_1000) / len(write_latencies_1000)
        
        # Add the data to the dictionary
        data["Compression Location"].extend([location, location])
        data["Producer Rate"].extend([100, 1000])
        data["Compression Ratio"].extend([avg_compression_ratio_100, avg_compression_ratio_1000])
        data["aggregatedPublishLatency95pct"].extend([avg_write_latency_100, avg_write_latency_1000])



In [ ]:

df = pd.DataFrame(data)

# Create a plot
plt.figure(figsize=(9,5))

# Define the colors and markers for each compression location
colors = {'No Compression': 'blue', 'Nexus Compression': 'red'}
markers = {'100': 'o', '1000': '^'}

# Plot the data
for location in df['Compression Location'].unique():
    location_df = df[df['Compression Location'] == location]
    for rate in location_df['Producer Rate'].unique():
        rate_df = location_df[location_df['Producer Rate'] == rate]
        plt.plot(rate_df['Compression Ratio'], rate_df['aggregatedPublishLatency95pct'], marker=markers[str(rate)], color=colors[location], linestyle='None', label=f'{location} ({rate} e/s)')

# Add a legend
plt.legend(title='Compression Location - Producer Rate', bbox_to_anchor=(1, 1.02), loc='upper left')

# Set the x and y axis labels
plt.xlabel('Data Compression Ratio')
plt.ylabel('p95 Write Latency (ms)')

# Set the title
plt.title('Compression Ratio vs Write Latency (Pravega)')

# Show the plot
plt.show()